# Hands-on Session 2: Running DGAT Protein Inference Workflows

**Time:** 11:05-11:35  
**Lead:** Zeyuan

Goals:
- Prepare transcript and spatial inputs for DGAT-style inference.
- Run the official DGAT workflow or load pretrained DGAT predictions.
- Generate inferred spatial protein landscapes.
- Visualize predicted protein expression patterns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad, find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table, run_demo_dgat_inference, run_official_dgat_prediction
from dgat_tutorial.plotting import plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
processed_dir = repo_root / "data" / "processed"
fig_dir = repo_root / "results" / "figures"
processed_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

## Load Tutorial Data

In [ ]:
dgat_h5ad_pair = find_dgat_h5ad_pair(data_dir)
dgat_h5ad = find_dgat_h5ad(data_dir)
dataset = load_tutorial_data(data_dir, allow_demo=True)
if dgat_h5ad_pair:
    print(f"Loaded DGAT AnnData files: RNA={dgat_h5ad_pair[0]}, ADT={dgat_h5ad_pair[1]}")
else:
    print(f"Loaded DGAT AnnData file: {dgat_h5ad}" if dgat_h5ad else "Loaded synthetic fallback data")

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

print(transcripts.shape, proteins.shape)

## Prepare DGAT Inputs

For the live tutorial, use the official DGAT repository as the source of model code: https://github.com/osmanbeyoglulab/DGAT

The original DGAT workflow reads `.h5ad` data directly. This notebook keeps the same data source for exploration and uses `data/raw/dgat_predictions.csv` only as a precomputed prediction fallback for committee review or participant backup.

In [ ]:
common_spots = spots.index.intersection(transcripts.index).intersection(proteins.index)
spots = spots.loc[common_spots]
transcripts = transcripts.loc[common_spots]
proteins = proteins.loc[common_spots]

spots.to_csv(processed_dir / "spots_aligned.csv")
transcripts.to_csv(processed_dir / "transcripts_aligned.csv")
proteins.to_csv(processed_dir / "proteins_aligned.csv")

print(f"Prepared {len(common_spots)} aligned spots")

## Run or Load DGAT Predictions

Preferred path: run the official DGAT `protein_predict(...)` workflow from the cloned DGAT repository and downloaded pretrained models.

Fallback path: if `data/raw/dgat_predictions.csv` already exists, load it. The small ridge-regression demo fallback is only for environment checks.

In [ ]:
run_official_dgat = False  # Set True after cloning external/DGAT and downloading DGAT_pretrained_models.
prediction_path = data_dir / "dgat_predictions.csv"
dgat_repo_dir = repo_root / "external" / "DGAT"
model_save_dir = repo_root / "external" / "DGAT_assets" / "DGAT_pretrained_models"
pyg_data_dir = processed_dir / "dgat_pyg"

if prediction_path.exists():
    predicted_proteins = load_prediction_table(str(prediction_path))
    print("Loaded precomputed DGAT predictions")
elif run_official_dgat:
    rna_h5ad = dgat_h5ad_pair[0] if dgat_h5ad_pair else dgat_h5ad
    predicted_proteins = run_official_dgat_prediction(
        rna_h5ad_path=rna_h5ad,
        dgat_repo_dir=dgat_repo_dir,
        model_save_dir=model_save_dir,
        pyg_data_dir=pyg_data_dir,
    )
    print("Generated predictions with official DGAT protein_predict")
else:
    predicted_proteins = run_demo_dgat_inference(transcripts, proteins)
    print("Generated demo DGAT-like predictions. Set run_official_dgat=True for pretrained DGAT.")

predicted_proteins.to_csv(processed_dir / "predicted_proteins.csv")
predicted_proteins.head()

## Visualize Inferred Protein Landscapes

In [ ]:
proteins_to_plot = list(predicted_proteins.columns[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 8))

for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(spots, predicted_proteins[protein], f"Predicted {protein}", ax=ax)

plt.tight_layout()
plt.savefig(fig_dir / "session02_predicted_proteins.png", dpi=160)
plt.show()

## Checkpoint

Participants should now have generated a predicted protein matrix and spatial plots that can be interpreted in Session 3.